# Final Round Standalone Verification Notebook

This notebook validates consistency of packaged final evidence files only.
It does not depend on external submission folders.


In [ ]:
from pathlib import Path
import json
import pandas as pd

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    candidate = NB_DIR / 'final_round_clean_submission' / 'notebooks'
    if candidate.exists():
        NB_DIR = candidate

BASE_DIR = NB_DIR.parent
EVIDENCE_DIR = BASE_DIR / 'evidence'
print('Base dir:', BASE_DIR)


In [ ]:
evidence = json.loads((EVIDENCE_DIR / 'evidence_latest_run.json').read_text(encoding='utf-8'))
benchmark = json.loads((EVIDENCE_DIR / 'benchmark_summary.json').read_text(encoding='utf-8'))
full_df = pd.read_csv(EVIDENCE_DIR / 'benchmark_full_sample_metrics.csv')
subsample_df = pd.read_csv(EVIDENCE_DIR / 'benchmark_subsample_summary.csv')
print('Loaded evidence files successfully.')


In [ ]:
m = evidence.get('metrics', {})
metrics_table = pd.DataFrame([
    {'metric': 'ensemble_pr_auc', 'value': float(m.get('ensemble_pr_auc', 0.0))},
    {'metric': 'ensemble_roc_auc', 'value': float(m.get('ensemble_roc_auc', 0.0))},
    {'metric': 'catboost_pr_auc', 'value': float(m.get('catboost_pr_auc', 0.0))},
    {'metric': 'deep_learning_pr_auc', 'value': float(m.get('deep_learning_pr_auc', 0.0))},
])
metrics_table


In [ ]:
full = benchmark.get('full_sample_metrics', {})
ens = full.get('latest_ensemble', {})
tsf = full.get('timesfm_proxy', {})
delta = full.get('delta', {})
benchmark_table = pd.DataFrame([
    {'model':'latest_ensemble', 'pr_auc': float(ens.get('pr_auc',0.0)), 'roc_auc': float(ens.get('roc_auc',0.0)), 'brier': float(ens.get('brier_score',0.0))},
    {'model':'timesfm_proxy', 'pr_auc': float(tsf.get('pr_auc',0.0)), 'roc_auc': float(tsf.get('roc_auc',0.0)), 'brier': float(tsf.get('brier_score',0.0))},
])
delta_table = pd.DataFrame([
    {'delta_metric':'pr_auc_ensemble_minus_timesfm', 'value': float(delta.get('pr_auc_ensemble_minus_timesfm',0.0))},
    {'delta_metric':'roc_auc_ensemble_minus_timesfm', 'value': float(delta.get('roc_auc_ensemble_minus_timesfm',0.0))},
    {'delta_metric':'brier_timesfm_minus_ensemble', 'value': float(delta.get('brier_timesfm_minus_ensemble',0.0))},
])
benchmark_table
delta_table


In [ ]:
required_files = [
    BASE_DIR / 'presentation' / 'AesCodeNexus_Final_Round_Deck.pptx',
    BASE_DIR / 'scripts' / 'FINAL_DEMO_VIDEO_SCRIPT.md',
    BASE_DIR / 'scripts' / 'PORTAL_SUBMISSION_CHECKLIST.md',
    EVIDENCE_DIR / 'evidence_latest_run.json',
    EVIDENCE_DIR / 'benchmark_summary.json',
    EVIDENCE_DIR / 'benchmark_full_sample_metrics.csv',
]
checks = pd.DataFrame([{'file': str(p.relative_to(BASE_DIR)).replace('\\','/'), 'exists': p.exists()} for p in required_files])
checks


In [ ]:
alignment = benchmark.get('alignment', {})
pass_flags = {
    'has_run_name': bool(evidence.get('run_name')),
    'labels_match': bool(alignment.get('labels_match', False)),
    'positive_pr_delta': float(delta.get('pr_auc_ensemble_minus_timesfm', 0.0)) > 0,
    'all_required_files_present': bool(checks['exists'].all()),
}
overall_pass = all(pass_flags.values())
report = {
    'run_name': evidence.get('run_name', ''),
    'pass_flags': pass_flags,
    'overall_pass': overall_pass,
    'ensemble_pr_auc': float(m.get('ensemble_pr_auc', 0.0)),
    'benchmark_pr_delta': float(delta.get('pr_auc_ensemble_minus_timesfm', 0.0)),
}
out = BASE_DIR / 'notebooks' / 'reproducibility_report.json'
out.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Overall pass:', overall_pass)
print('Report written:', out)
report


## Submission Note

Upload this notebook to Kaggle or Colab and keep the link public before form submission.
